# LAB 4 — Pipeline Naive RAG Completo

## Objetivo

Implementar um **RAG pipeline ponta-a-ponta** integrando:
1. Docling: Ingestão de PDFs jurídicos
2. RecursiveCharacterTextSplitter: Chunking jurídico
3. BGE-M3: Embeddings (HuggingFace)
4. FAISS: Índice vetorial local
5. vLLM: Servidor LLM (Llama 3.1 8B)
6. LangChain LCEL: Orquestração completa

## Stack

- **Framework**: LangChain (LCEL)
- **Embeddings**: BAAI/bge-m3 (1024d, multilíngue)
- **Vectorstore**: FAISS (local, rápido)
- **LLM**: Llama 3.1 8B via vLLM
- **Python**: 3.11+
- **Ambiente**: Google Colab

## Diagrama

```
PDFs → Docling → Markdown → Chunking → BGE-M3 → FAISS → Retriever → vLLM → Resposta
```

In [ ]:
# Instalação de dependências
import subprocess
import sys

packages = [
    'langchain', 'langchain-community', 'langchain-core', 'langchain-text-splitters',
    'sentence-transformers', 'faiss-cpu', 'opensearch-py', 'docling', 'pypdf',
    'vllm', 'openai', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'python-dotenv'
]

print("📦 Instalando dependências...")
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ Instalação completa!")

In [ ]:
# Criar corpus jurídico de teste
import os
from pathlib import Path

PDF_DIR = '/tmp/corpus_juridico'
os.makedirs(PDF_DIR, exist_ok=True)

# Criar texto de exemplo para ingestão (simulando PDFs)
corpus_exemplo = {
    'Lei 11.343': '''
LEI Nº 11.343, DE 23 DE AGOSTO DE 2006
Institui o Sistema Nacional de Políticas Públicas sobre Drogas.

Art. 28. Quem adquirir, guardar, tiver em depósito, transportar ou trouxer consigo,
para consumo pessoal, drogas sem autorização será submetido às seguintes penas:
I - advertência sobre os efeitos das drogas;
II - prestação de serviços à comunidade;
III - medida educativa de comparecimento a programa educativo.

Art. 33. Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender,
distribuir, entregar a qualquer título, guardar, ter em depósito, transportar, trazer
consigo, ainda que gratuitamente, sem autorização ou em desacordo com determinação
legal ou regulamentar, matéria-prima, insumo ou produto químico destinado à elaboração
de drogas ilícitas.
Pena - reclusão de 5 a 15 anos e pagamento de 500 a 1.500 salários mínimos.

§ 4º As penas poderão ser reduzidas de um sexto a dois terços, vedada a substituição
por penas restritivas de direitos, desde que o agente colabore efetivamente com a
justiça na investigação e processos penais.
''',
    'Acórdão HC': '''
HABEAS CORPUS Nº 123.456/SP

EMENTA:
Habeas Corpus. Prisão preventiva. Fundamentação adequada. Presença de fumus comissi
delicti e periculum libertatis. Incidência do art. 312 do CPP. Ordem denegada.

RELATÓRIO:
Trata-se de habeas corpus impetrado contra decisão do Tribunal de Justiça do Estado
de São Paulo que manteve a prisão preventiva decretada em desfavor do paciente,
acusado da prática de crime previsto no art. 33 da Lei nº 11.343/2006.

DISPOSITIVO:
Pelo exposto, DENEGO A ORDEM de habeas corpus.
''',
    'Relatório Inteligência': '''
RELATÓRIO DE INTELIGÊNCIA - 1º TRIMESTRE 2025

RESUMO EXECUTIVO:
O presente relatório apresenta análise consolidada das ocorrências registradas no
período de janeiro a março de 2025. Os dados indicam aumento de 12,3% em roubos a
mão armada e redução de 8,7% em furtos.

ANÁLISE POR TIPO DE CRIME:
Homicídio doloso: 446 ocorrências (-5,2%)
Roubo a mão armada: 1.722 ocorrências (+12,3%)
Furto simples: 3.550 ocorrências (-8,7%)
Tráfico de drogas: 260 ocorrências (-14,6%)

OPERAÇÕES ESPECIAIS:
Durante o período, foram deflagradas 18 operações estruturadas. Destaca-se a
Operação "Hércules" de combate ao tráfico de cocaína na zona leste, que resultou
na apreensão de 234 kg de cocaína e 456 kg de maconha.
'''
}

print("✅ Corpus jurídico carregado (3 documentos)")

In [ ]:
# Chunking jurídico com LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def chunkar_corpus(corpus):
    """Divide corpus em chunks respeitando estrutura jurídica."""
    separadores = [
        "\n\nArt\.",
        "\n\n§",
        "\nI -",
        "\nII -",
        "\n\n",
        "\n",
        ". ",
        " ",
    ]
    
    splitter = RecursiveCharacterTextSplitter(
        separators=separadores,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    
    todos_chunks = []
    for nome, texto in corpus.items():
        chunks_texto = splitter.split_text(texto)
        for chunk_id, chunk_texto in enumerate(chunks_texto):
            doc = Document(
                page_content=chunk_texto,
                metadata={
                    'fonte': nome,
                    'tipo_documento': 'juridico',
                    'chunk_id': chunk_id,
                    'chars': len(chunk_texto),
                }
            )
            todos_chunks.append(doc)
    
    return todos_chunks

chunks = chunkar_corpus(corpus_exemplo)
print(f"✅ Chunking completo: {len(chunks)} chunks")

In [ ]:
# Embeddings com BGE-M3
from langchain_community.embeddings import HuggingFaceEmbeddings
import time

print("📥 Carregando modelo BGE-M3...")
tempo_inicio = time.time()

model_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

tempo = time.time() - tempo_inicio
print(f"✅ Modelo carregado em {tempo:.2f}s")
print(f"\n📊 Info: 1024 dimensões, multilíngue, multi-granularidade")

In [ ]:
# Gerar embeddings para todos os chunks
print("🔄 Gerando embeddings...")

chunks_textos = [c.page_content for c in chunks]
embeddings = model_embeddings.embed_documents(chunks_textos)

print(f"✅ {len(embeddings)} embeddings gerados")
print(f"   Dimensão: {len(embeddings[0])}")

In [ ]:
# Criar índice FAISS
from langchain_community.vectorstores import FAISS

print("📦 Criando índice FAISS...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=model_embeddings,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print(f"✅ Índice criado com {len(chunks)} vetores")

In [ ]:
# Configurar vLLM ou OpenAI
from langchain_openai import ChatOpenAI
import os

print("🔌 Configurando LLM...")

# Tentar vLLM local
try:
    import requests
    requests.get('http://localhost:8000/v1/models', timeout=1)
    print("   ✅ vLLM detectado em localhost:8000")
    
    llm = ChatOpenAI(
        base_url="http://localhost:8000/v1",
        api_key="dummy-key",
        model="meta-llama/Llama-3.1-8B-Instruct",
        temperature=0.7,
        max_tokens=512,
    )
except:
    print("   ⚠️  vLLM não disponível, usando OpenAI API...")
    api_key = os.getenv('OPENAI_API_KEY', 'sk-test')
    
    llm = ChatOpenAI(
        api_key=api_key,
        model="gpt-4o-mini",
        temperature=0.7,
        max_tokens=512,
    )

print("✅ LLM configurada")

In [ ]:
# Criar prompt template jurídico
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def formatar_contexto(docs):
    """Formata documents em contexto legível com citações."""
    partes = []
    for i, doc in enumerate(docs, 1):
        fonte = doc.metadata.get('fonte', 'N/A')
        bloco = f"[Fonte {i}] {fonte}:\n{doc.page_content}"
        partes.append(bloco)
    return "\n\n".join(partes)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", 
     "Você é um especialista jurídico. Responda APENAS com base no contexto fornecido. "
     "Sempre cite a fonte [Fonte N]. Se não encontrar, diga: 'Não consta nos documentos'.\n\n"
     "CONTEXTO:\n{context}"),
    ("human", "{question}"),
])

# Criar RAG chain (LCEL)
rag_chain = (
    {
        "context": retriever | RunnableLambda(formatar_contexto),
        "question": RunnablePassthrough(),
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain criada com LCEL")

In [ ]:
# Testar RAG pipeline
import time

def perguntar(query):
    """Executa query no RAG pipeline com debugging."""
    print(f"\n❓ Pergunta: {query}")
    print("="*80)
    
    # Retrieval
    tempo_inicio = time.time()
    docs = retriever.invoke(query)
    tempo_retrieval = time.time() - tempo_inicio
    
    print(f"\n🔎 [Retrieval] {len(docs)} chunks em {tempo_retrieval*1000:.1f}ms")
    for i, doc in enumerate(docs, 1):
        preview = doc.page_content[:60].replace('\n', ' ')
        print(f"   [{i}] {doc.metadata['fonte']}: \"{preview}...\"")
    
    # Geração
    print(f"\n💬 [Geração]...")
    tempo_inicio = time.time()
    
    try:
        resposta = rag_chain.invoke(query)
        tempo_gen = time.time() - tempo_inicio
        
        print(f"✅ Resposta em {tempo_gen:.2f}s\n")
        print(resposta)
        
    except Exception as e:
        print(f"❌ Erro: {e}")

# Testar
queries = [
    "Qual é a pena para tráfico de drogas?",
    "Quando a prisão preventiva é cabível?",
    "Qual foi o número de homicídios em 2025?",
]

for q in queries:
    perguntar(q)
    print()

In [ ]:
# Salvar índice para próxima sessão
INDEX_DIR = '/tmp/rag_index'
import os

os.makedirs(INDEX_DIR, exist_ok=True)

vectorstore.save_local(f"{INDEX_DIR}/faiss_index")
print(f"✅ Índice salvo em {INDEX_DIR}")
print(f"\n📖 Para carregar em nova sessão:")
print(f"""  vectorstore = FAISS.load_local(
      '{INDEX_DIR}/faiss_index',
      model_embeddings,
      allow_dangerous_deserialization=True
  )""")

In [ ]:
# Exercício: Adicionar novo documento
print("""
📚 EXERCÍCIO: Adicione um novo documento ao RAG

Passos:
1. novo_corpus = {'Seu doc': 'seu_texto_aqui'}
2. novos_chunks = chunkar_corpus(novo_corpus)
3. vectorstore.add_documents(novos_chunks)
4. perguntar('pergunta relevante ao novo doc')
5. vectorstore.save_local('/tmp/rag_index/faiss_index')

Pronto! Seu RAG agora conhece o novo documento.
""")